In [2]:
# cell 1 : Read silver_sales 
df= spark.read.format('delta').load('Tables/dbo/silver_sales') 
print(f'silver_sales rows: {df.count()}')


StatementMeta(, b01cf1f8-2133-4eae-96ae-b33c0fab5f98, 3, Finished, Available, Finished, False)

silver_sales rows: 1268


In [4]:
# cell 2: build dim_customer
from pyspark.sql.functions import monotonically_increasing_id
dim_customer= df.select('customer_name', 'region')\
               .distinct()\
               .withColumn('customer_key' , monotonically_increasing_id()+1)
# reoder columns : key column first 
dim_customer = dim_customer.select('customer_key','customer_name', 'region')

dim_customer.write.format('delta').mode('overwrite')\
       .saveAsTable('dim_customer')

print(f'dim_customer rows : {dim_customer.count()}')  
display(dim_customer.limit(5))          

StatementMeta(, b01cf1f8-2133-4eae-96ae-b33c0fab5f98, 5, Finished, Available, Finished, False)

dim_customer rows : 93


SynapseWidget(Synapse.DataFrame, 8f928bc0-fcc2-4a33-8e99-6fe016e8200f)

In [5]:
# cell 3: build dim_product
from pyspark.sql.functions import monotonically_increasing_id
dim_product= df.select('product_category')\
               .distinct()\
               .withColumn('product_key' , monotonically_increasing_id()+1)
# reoder columns : key column first 
dim_product = dim_product.select('product_key','product_category')

dim_product.write.format('delta').mode('overwrite')\
       .saveAsTable('dim_product')

print(f'dim_product rows : {dim_product.count()}')  
display(dim_product)          

StatementMeta(, b01cf1f8-2133-4eae-96ae-b33c0fab5f98, 6, Finished, Available, Finished, False)

dim_product rows : 3


SynapseWidget(Synapse.DataFrame, f1d1b362-db4c-4a64-8475-09d922517bd4)

In [7]:
# cell 4: build dim_region
from pyspark.sql.functions import monotonically_increasing_id
dim_region= df.select('region')\
               .distinct()\
               .withColumn('region_key' , monotonically_increasing_id()+1)
# reoder columns : key column first 
dim_region = dim_region.select('region_key','region')

dim_region.write.format('delta').mode('overwrite')\
       .saveAsTable('dim_region')

print(f'dim_region rows : {dim_region.count()}')  
display(dim_region)          

StatementMeta(, b01cf1f8-2133-4eae-96ae-b33c0fab5f98, 8, Finished, Available, Finished, False)

dim_region rows : 4


SynapseWidget(Synapse.DataFrame, dc7edfc7-8d4c-4dd8-a605-dc5d095fb688)

In [13]:
# cell 5: build fact_sales by joining silver_sales to each dimension 
from pyspark.sql.functions import col , to_date , date_format

# re-read silver and all dimensions
df_silver = spark.read.format('delta').load('Tables/dbo/silver_sales')
df_customer = spark.read.format('delta').load('Tables/dbo/dim_customer')
df_product = spark.read.format('delta').load('Tables/dbo/dim_product')
df_region = spark.read.format('delta').load('Tables/dbo/dim_region')

# join to resolve surrogate_keys
fact= df_silver\
    .join(df_customer,
    (df_silver.customer_name == df_customer.customer_name) &
    (df_silver.region == df_customer.region),
    'left')\
    .join(df_product,
    (df_silver.product_category == df_product.product_category),
     'left')\
    .join(df_region,
    (df_silver.region == df_region.region),
     'left')
# build the date surrogate key(YYYYMMDD) interger
fact = fact.withColumn(
    'order_date_key',
    date_format(to_date(col('order_date'),'yyyy-MM-dd'),'yyyyMMdd').cast('int')
)       
# select only the fact table column 
fact_sales= fact.select(
    col('order_id'),
    col('order_date_key'),
    col('customer_key'),
    col('product_key'),
    col('region_key'),
    col('revenue'),
    col('quantity'),
    col('revenue_USD'))

#  write the fact_sales table 
fact_sales.write.format('delta').mode('overwrite')\
       .saveAsTable('fact_sales')

print(f'fact_sales rows : {fact_sales.count()}')  
display(fact_sales.limit(5))          

StatementMeta(, b01cf1f8-2133-4eae-96ae-b33c0fab5f98, 14, Finished, Available, Finished, False)

fact_sales rows : 1268


SynapseWidget(Synapse.DataFrame, b4d2fcfa-f70e-42cc-ae78-b9e65e350844)